In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/credit_risk_dataset.csv')

# Hapus baris duplikat untuk mencegah overfitting
df = df.drop_duplicates()

print("Jumlah baris awal setelah hapus duplikat:", df.shape[0])
df.head()

Jumlah baris awal setelah hapus duplikat: 32416


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [ ]:
# Membuang baris dengan usia > 100 tahun
df = df[df['person_age'] <= 100]

# Membuang baris dengan lama bekerja >= 123 tahun 
df = df[df['person_emp_length'] <= 60]

print("Jumlah baris setelah membuang outlier ekstrem:", df.shape[0])

Jumlah baris setelah membuang outlier ekstrem: 31522


In [3]:
# Mengisi missing values loan_int_rate menggunakan median berdasarkan loan_grade
df['loan_int_rate'] = df['loan_int_rate'].fillna(
    df.groupby('loan_grade')['loan_int_rate'].transform('median')
)

# Untuk person_emp_length, kita bisa imputasi dengan median keseluruhan
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())

# Cek kembali apakah masih ada yang kosong
print("Sisa missing values setelah pembersihan:")
print(df.isnull().sum())

Sisa missing values setelah pembersihan:
person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64


In [4]:
# Mapping loan_grade sesuai tingkat risiko (A = 0 paling aman, G = 6 paling berisiko)
grade_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
df['loan_grade'] = df['loan_grade'].map(grade_mapping)

# Mapping riwayat gagal bayar di biro kredit (N = 0, Y = 1)
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'N': 0, 'Y': 1})

df[['loan_grade', 'cb_person_default_on_file']].head()

,loan_grade,cb_person_default_on_file
1,1,0
2,2,0
3,2,0
4,2,1
5,0,0


In [5]:
# One-Hot Encoding untuk variabel kategori nominal
df_encoded = pd.get_dummies(
    df, 
    columns=['person_home_ownership', 'loan_intent'], 
    drop_first=True, # Mencegah jebakan multikolinearitas (dummy variable trap)
    dtype=int
)

print("Jumlah kolom setelah encoding:", df_encoded.shape[1])
df_encoded.head()

Jumlah kolom setelah encoding: 18


,person_age,person_income,person_emp_length,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
1,21,9600,5.0,1,1000,11.14,0,0.10,0,2,0,1,0,1,0,0,0,0
2,25,9600,1.0,2,5500,12.87,1,0.57,0,3,0,0,0,0,0,1,0,0
3,23,65500,4.0,2,35000,15.23,1,0.53,0,2,0,0,1,0,0,1,0,0
4,24,54400,8.0,2,35000,14.27,1,0.55,1,4,0,0,1,0,0,1,0,0
5,21,9900,2.0,0,2500,7.14,1,0.25,0,2,0,1,0,0,0,0,0,1


In [6]:
from sklearn.model_selection import train_test_split

# Memisahkan fitur (X) dan target (y)
X = df_encoded.drop('loan_status', axis=1)
y = df_encoded['loan_status']

# Cek proporsi kelas target (apakah data tidak seimbang / imbalanced?)
print("Proporsi kelas loan_status:")
print(y.value_counts(normalize=True) * 100)

# Menyimpan dataset yang sudah bersih dan ter-encode ke folder data/
df_encoded.to_csv('../data/credit_risk_cleaned.csv', index=False)
print("\nDataset bersih berhasil disimpan ke '../data/credit_risk_cleaned.csv'!")

Proporsi kelas loan_status:
loan_status
0    78.405558
1    21.594442
Name: proportion, dtype: float64

Dataset bersih berhasil disimpan ke '../data/credit_risk_cleaned.csv'!
